# 추출 캐시 테스트 (작업지시서 §6)

지금까지는 방출기(cells·references·diagnostics·meta)가 **무엇을** 만드는지를 봤습니다.
이번 `cache.py`는 **"이미 만든 걸 또 만들지 말자"**를 판정하는 부품입니다.

산출물 본체는 DB가 아니라 **파일시스템 폴더**에 둡니다:

```
converted/
├── _index.json                          ← 목록표(대장)
└── {원본stem_slug}_{sha256 앞 12자}/     ← 원본 1개 = 폴더 1개
```

캐시 키는 **`sha256(파일) + converter_version`**(§6). 판정 규칙(오너 결정):
**같은 sha256 + 같은 converter_version + 폴더가 실재** — 셋 다 맞아야 `hit`(재사용),
하나라도 어긋나면 `miss`(재생성).

- **파일이 바뀌면** sha가 바뀌어 애초에 폴더명부터 달라집니다(옛 패키지와 공존).
- **코드만 올라가면** 폴더명은 같으니 `_index.json`에 적힌 버전을 대조해 잡습니다.
- `_index.json`의 `generated_at`(최종생성시각)만 가변값 — 재현성 비교에서 뺍니다.
- 승계 규칙(§6)은 해석 계층(M3)이 생긴 뒤에 붙입니다. 지금은 버전이 다르면 재생성.

확인하는 것:
1. **slug 규칙** — 공백→`_`, 경로 위험 문자 제거, 한글 유지
2. **폴더명** — 실파일 sha(골든 `9c7aa730…`)로 이름이 결정되는지
3. **probe 판정** — hit 1가지 + miss 5가지 사유가 정확한지
4. **색인 항목** — 스키마 필드 순서 + M1은 미주석 고정
5. **결정론** — 삽입 순서가 달라도, 재작성해도 색인 바이트가 같은지

> 실행 전 커널이 이 프로젝트의 `.venv`인지 확인하세요.

In [1]:
# 0. 준비 — 경로 자동 판별과 import
import json, shutil, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():   # tests/ 안에서 열었을 때
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.cache import (
    load_index, package_dirname, probe, record, save_index, slugify)
from excel_to_skill.meta import _converter_version, _source_sha256

RAW = ROOT / "workingpaper_raw"            # 실조서 코퍼스 (git 밖)
SANDBOX = ROOT / "tests" / "_output" / "cache_nb"   # 노트북 산출물 (git 밖)
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
OUT = SANDBOX / "converted"

CV = _converter_version()
print("프로젝트 루트:", ROOT)
print("converter_version:", CV)

프로젝트 루트: /home/shin/Project/Excel_to_Skill
converter_version: 0.1.0


## 1. slug 규칙 — 폴더·시트 이름을 안전하게

파일명·시트명을 그대로 폴더명에 쓸 수 없습니다(경로에 못 쓰는 문자·공백).
§4.0 규칙: **공백류는 `_`로, 경로 위험 문자(`/ \\ : * ? " < > |`)는 제거,
한글은 그대로**. 다 지워지면 `untitled`로 떨어집니다.

In [2]:
cases = {
    "감사계약 1100~1300":       "감사계약_1100~1300",
    'a/b:c*d?"e<f>g|h':          "abcdefgh",
    "  공백   여럿  ":            "공백_여럿",
    "...점만...":                 "점만",
    "///":                        "untitled",
}
print(f"{'입력':<24} {'출력':<20} 판정")
ok = True
for src, want in cases.items():
    got = slugify(src)
    ok &= got == want
    print(f"{src!r:<24} {got!r:<20} {'O' if got == want else 'X'} (기대 {want!r})")
assert ok, "slug 규칙 불일치"
print("\n결과: 통과")

입력                       출력                   판정
'감사계약 1100~1300'         '감사계약_1100~1300'     O (기대 '감사계약_1100~1300')
'a/b:c*d?"e<f>g|h'       'abcdefgh'           O (기대 'abcdefgh')
'  공백   여럿  '            '공백_여럿'              O (기대 '공백_여럿')
'...점만...'               '점만'                 O (기대 '점만')
'///'                    'untitled'           O (기대 'untitled')

결과: 통과


## 2. 폴더명 — 실파일 지문으로 이름이 정해지나

폴더명은 `{원본stem_slug}_{sha256 앞 12자}`. 감사계약 파일로 실제 이름을 만들고,
sha가 골든(`9c7aa730…`)과 맞는지 봅니다. **같은 파일이면 언제나 같은 폴더명**입니다.

In [3]:
p = next(RAW.glob("*1100~1300*"))
sha = _source_sha256(p)
dirname = package_dirname(p, sha)
print("원본 :", p.name)
print("sha  :", sha)
print("폴더 :", dirname)
assert sha.startswith("9c7aa730172a"), "골든 sha 불일치"
assert dirname.endswith("_" + sha[:12])
assert dirname == package_dirname(p, sha)   # 결정론
print("\n결과: 통과 (폴더명 = stem_slug + sha 앞 12자, 재현)")

원본 : 감사조서서식_1100~1300 감사계약.xlsx
sha  : 9c7aa730172a7c9aa1079f5b513accc4856d4ca831ef25ba2bbaf7455daba64e
폴더 : 감사조서서식_1100~1300_감사계약_9c7aa730172a

결과: 통과 (폴더명 = stem_slug + sha 앞 12자, 재현)


## 3. probe 판정 — hit 1가지 + miss 5가지 사유

캐시 조회는 `hit` 여부와 **사유**를 함께 돌려줍니다(cli가 왜 재생성하는지 로그로
밝히려고). 아래 순서로 상황을 하나씩 만들며 판정이 맞는지 봅니다:

1. 색인 비었음 → `absent`
2. 폴더+색인 있음 → **`match` (hit)**
3. converter_version 다름 → `version_changed`
4. 폴더 삭제됨 → `folder_missing`
5. `--force` → `force`
6. 색인 sha 위조(12자 접두 충돌 방어) → `sha_mismatch`

In [4]:
def line(n, desc, pr):
    print(f"  {n}) {desc:<20} → hit={str(pr.hit):<5} reason={pr.reason}")

# 1) absent
pr = probe(OUT, p, converter_version=CV); line(1, "색인 비었음", pr)
assert not pr.hit and pr.reason == "absent"

# 2) match — 폴더 만들고 색인 기록
pkg = OUT / dirname
pkg.mkdir(parents=True)
(pkg / "meta.json").write_text("{}", encoding="utf-8")
record(OUT, p, sha256=sha, converter_version=CV, generated_at="2026-07-09T00:00:00Z")
pr = probe(OUT, p, converter_version=CV); line(2, "폴더+색인 있음", pr)
assert pr.hit and pr.reason == "match" and pr.package_path == pkg

# 3) version_changed
pr = probe(OUT, p, converter_version="9.9.9"); line(3, "버전 상이", pr)
assert not pr.hit and pr.reason == "version_changed"

# 4) folder_missing
shutil.rmtree(pkg)
pr = probe(OUT, p, converter_version=CV); line(4, "폴더 삭제됨", pr)
assert not pr.hit and pr.reason == "folder_missing"

# 5) force
pkg.mkdir(parents=True)
pr = probe(OUT, p, converter_version=CV, force=True); line(5, "--force", pr)
assert not pr.hit and pr.reason == "force"

# 6) sha_mismatch
idx = load_index(OUT); idx["entries"][dirname]["sha256"] = "0" * 64; save_index(OUT, idx)
pr = probe(OUT, p, converter_version=CV); line(6, "sha 접두충돌", pr)
assert not pr.hit and pr.reason == "sha_mismatch"

print("\n결과: 통과 (hit 1 · miss 5사유 정확)")

  1) 색인 비었음               → hit=False reason=absent
  2) 폴더+색인 있음             → hit=True  reason=match
  3) 버전 상이                → hit=False reason=version_changed
  4) 폴더 삭제됨               → hit=False reason=folder_missing
  5) --force              → hit=False reason=force
  6) sha 접두충돌             → hit=False reason=sha_mismatch

결과: 통과 (hit 1 · miss 5사유 정확)


## 4. 색인 항목 — 스키마 필드 순서 + M1 미주석 고정

`_index.json`의 한 항목은 §6이 정한 7필드입니다(원본명·sha256·경로·버전·주석키·
리뷰·시각). 해석 계층이 아직 없으니 `annotation_key`·`review_status`는 `null`,
`generated_at`만 가변값입니다.

In [5]:
# 깨끗한 색인으로 다시 기록
shutil.rmtree(OUT, ignore_errors=True)
entry = record(OUT, p, converter_version=CV, generated_at="2026-07-09T00:00:00Z")
print(json.dumps(entry, ensure_ascii=False, indent=2))

want = ["source_filename", "sha256", "package_path", "converter_version",
        "annotation_key", "review_status", "generated_at"]
print("\n필드 순서 일치:", list(entry) == want)
assert list(entry) == want
assert entry["annotation_key"] is None and entry["review_status"] is None
assert entry["source_filename"] == p.name
print("annotation_key/review_status 미주석(None):", True)
print("결과: 통과")

{
  "source_filename": "감사조서서식_1100~1300 감사계약.xlsx",
  "sha256": "9c7aa730172a7c9aa1079f5b513accc4856d4ca831ef25ba2bbaf7455daba64e",
  "package_path": "감사조서서식_1100~1300_감사계약_9c7aa730172a",
  "converter_version": "0.1.0",
  "annotation_key": null,
  "review_status": null,
  "generated_at": "2026-07-09T00:00:00Z"
}

필드 순서 일치: True
annotation_key/review_status 미주석(None): True
결과: 통과


## 5. 결정론 — 삽입 순서·재작성과 무관하게 색인이 같나

색인은 운영 파일이지만, 그래도 **삽입 순서가 달라도** 파일이 같아야 diff가 깨끗합니다
(entries를 폴더명 키로 정렬해서 씁니다). 같은 3개 항목을 순서만 바꿔 넣어 바이트가
같은지, 그리고 한 번 더 써도 같은지 봅니다. (`generated_at`은 고정값 주입)

In [6]:
r1 = SANDBOX / "c1"; r2 = SANDBOX / "c2"
files = sorted(RAW.glob("*.xls*"))[:3]
for f in files:            # 정순
    record(r1, f, converter_version=CV, generated_at="T")
for f in reversed(files):  # 역순
    record(r2, f, converter_version=CV, generated_at="T")

b1 = (r1 / "_index.json").read_bytes()
b2 = (r2 / "_index.json").read_bytes()
print("삽입 순서(정순 vs 역순) 달라도 바이트 동일:", b1 == b2)
assert b1 == b2

save_index(r1, load_index(r1))   # 읽고 다시 쓰기
print("재작성해도 바이트 동일:", (r1 / "_index.json").read_bytes() == b1)
assert (r1 / "_index.json").read_bytes() == b1
print("색인에 담긴 항목 수:", len(load_index(r1)["entries"]))
print("\n결과: 통과 (색인 결정론)")

삽입 순서(정순 vs 역순) 달라도 바이트 동일: True
재작성해도 바이트 동일: True
색인에 담긴 항목 수: 3

결과: 통과 (색인 결정론)
